<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 4 · Your Model in Production, with SHAP
### Train it in Colab. Ship it to Google Cloud. Explain every prediction it makes.

*Solo lab · about 2 hours · Colab · Dr. Wanik's cleaned Hartford cars, then your own Google Cloud bucket*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/Lab4_Model_In_Production.ipynb)

Since Lab 3 your pipeline scrapes cars, extracts them with **your** regex, and builds a clean master table every hour.
Tonight you put a **model you tuned yourself** into that pipeline, and make it **explain every prediction** with SHAP.

| Part | Time | What you do |
|---|---|---|
| **1. Load** | 10 min | Dr. Wanik's cleaned `listings_master.csv` (267 Hartford cars, after the A06 regex fix) |
| **2. Clean** | 15 min | The last mess: `Ford` vs `ford`, `chevy` vs `chevrolet`, placeholder mileage |
| **3. Baseline** | 10 min | The number to beat: a plain decision tree (the one your pipeline already trains every hour) |
| **4. Tune** | 30 min | Decision tree, random forest, or KNN, tuned with cross-validation. Pick a champion |
| **5. Explain** | 15 min | SHAP on your champion, before you ship it |
| **6. Ship** | 25 min | Save the model, Sync fork, upload it to your bucket, and watch `predict-shap` use it |
| **7. Read production** | 10 min | The predictions **and** SHAP values your pipeline saved for new cars |
| **8. Wrap up** | 5 min | Three short answers, submit |

**First: File → Save a copy in Drive.** This notebook opens from GitHub, so your edits disappear when you close the tab unless you work in your own copy.

**Stuck?** Ask Claude: paste the cell, the full error, and what you expected.

## The big picture
Your pipeline already scrapes (:00), extracts (:10 regex, :12 Gemini), combines (:15) and trains a basic tree (:20).
Tonight adds one more hourly step, **:25 · predict-shap**: it loads **your** model, predicts every new car, and saves
**why** (SHAP) next to **what** (the prediction).

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-labs/master/Module3/img/OPIM5512_M3_Pipeline_Overview.png" alt="How your pipeline works" width="100%">

# Part 1 · Load (10 min)

## 1a. Match production's library versions
Your model has to load inside a Google Cloud Function later, and a model saved with one version of scikit-learn may not
load in another. So we install the **exact** versions `predict-shap` uses. (If Colab asks you to **restart the
session**, click it, then run this notebook again from the top.)

In [ ]:
!pip install -q scikit-learn==1.6.1 shap==0.48.0

import sklearn
import shap
print("scikit-learn", sklearn.__version__, "| shap", shap.__version__, "(production uses 1.6.1 and 0.48.0)")

## 1b. Load Dr. Wanik's cleaned cars
This is the real `listings_master.csv` from Dr. Wanik's pipeline **after** the A06 fix: 267 Hartford cars, make and model
pulled by the Lab 3 regex. No sign-in needed.

In [ ]:
import io
import json
import os
import urllib.request

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab4_listings_master_snapshot.csv"
raw = pd.read_csv(DATA_URL, dtype={"post_id": str})
print(raw.shape[0], "cars,", raw.shape[1], "columns")
raw.head()

# Part 2 · Clean (15 min)

The regex fixed `Contact Information`, but real data always has a little mess left. Look at the make counts:
`ford` and `Ford` are the same car, and so are `chevy`, `chevrolet` and `Chevrolet`. A model would treat them as
different brands.

**Important:** `predict-shap` runs **this exact cleaning** on every new car before predicting. If you change it
here, your model and production stop agreeing.

In [ ]:
print(raw["make"].value_counts().head(12))

In [ ]:
FEATURES = ["year", "mileage", "make", "model"]
NICKNAMES = {"chevy": "chevrolet", "vw": "volkswagen", "mercedes": "mercedes-benz", "benz": "mercedes-benz"}


def clean_cars(df):
    out = df.copy()
    for col in ["price", "year", "mileage"]:
        out[col] = pd.to_numeric(out[col].astype(str).str.replace(r"[^\d.]+", "", regex=True), errors="coerce")
    for col in ["make", "model"]:
        out[col] = out[col].astype(str).str.strip().str.lower()
        out.loc[out[col].isin(["", "nan", "none"]), col] = np.nan
    out["make"] = out["make"].replace(NICKNAMES)
    return out


cars = clean_cars(raw)
print(cars["make"].value_counts().head(10))

## 2b. Flag the rows a model should not learn from
Same idea as Lab 3's review flags: an odd price, a placeholder mileage (`999999` means "I don't know"), or a very old
classic car priced by a collector. Flagged rows stay in the data, but they do **not** train the model.

In [ ]:
reasons = []
for i in range(len(cars)):
    row = cars.iloc[i]
    why = []
    if pd.isna(row["price"]) or row["price"] < 500 or row["price"] > 100000:
        why.append("odd price")
    if row["mileage"] in [0, 99999, 999999]:
        why.append("placeholder mileage")
    if pd.isna(row["year"]) or row["year"] < 1950:
        why.append("very old")
    reasons.append(", ".join(why))
cars["reason"] = reasons
cars["needs_review"] = cars["reason"] != ""

model_ready = cars[cars["needs_review"] == False].copy()
print(len(model_ready), "of", len(cars), "cars are ready for a model")
cars[cars["needs_review"]][["post_id", "year", "make", "model", "price", "mileage", "reason"]]

# Part 3 · Baseline (10 min)

Before tuning anything, write down the numbers to beat:
1. **Guess the median price for every car.** If your model cannot beat this, it learned nothing.
2. **A plain decision tree** with the same settings as `train-dt`, the model your pipeline already trains every hour.

We hold out 20% of the cars as a **test set** and never tune on them.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

X = model_ready[FEATURES]
y = model_ready["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "training cars |", len(X_test), "test cars")


def make_pipe(estimator):
    numeric = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2))])
    pre = ColumnTransformer([("num", numeric, ["year", "mileage"]), ("cat", categorical, ["make", "model"])])
    return Pipeline([("pre", pre), ("model", estimator)])


median_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.median()))
baseline = make_pipe(DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=42))
baseline.fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(X_test))
print("Guess the median:      MAE $" + format(round(median_mae), ","))
print("Baseline tree (train-dt): MAE $" + format(round(baseline_mae), ","))

`make_pipe` is the whole recipe in one object: fill missing values, scale the numbers, one-hot encode make and model
(rare ones are grouped as "infrequent"), then the model. **You save and ship the whole pipeline**, so production
gets the same preprocessing for free.

# Part 4 · Tune three ways (30 min)

`GridSearchCV` tries every combination in a grid, scores each one with **5-fold cross-validation** on the training
cars (MAE, lower is better), and keeps the best. Three model families, three grids. Run it: it takes a minute or two.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

candidates = {
    "decision_tree": (DecisionTreeRegressor(random_state=42),
                      {"model__max_depth": [3, 5, 8, 12, None], "model__min_samples_leaf": [1, 5, 10, 20]}),
    "random_forest": (RandomForestRegressor(random_state=42, n_jobs=-1),
                      {"model__n_estimators": [100, 300], "model__max_depth": [None, 10],
                       "model__min_samples_leaf": [1, 3, 5]}),
    "knn": (KNeighborsRegressor(),
            {"model__n_neighbors": [3, 5, 7, 11, 15], "model__weights": ["uniform", "distance"]}),
}

searches = {}
rows = []
for name in candidates:
    estimator, grid = candidates[name]
    search = GridSearchCV(make_pipe(estimator), grid, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    test_mae = mean_absolute_error(y_test, search.best_estimator_.predict(X_test))
    rows.append({"model": name, "cv_mae": round(-search.best_score_), "test_mae": round(test_mae),
                 "best_params": search.best_params_})

results = pd.DataFrame(rows).sort_values("cv_mae")
print("Baseline tree test MAE: $" + format(round(baseline_mae), ","))
results

**Pick your champion** by **cross-validation MAE** (`cv_mae`), not test MAE: the test set is your final exam, so you
do not choose with it. Type its name below (`decision_tree`, `random_forest` or `knn`).

In [ ]:
CHAMPION = "random_forest"     # <-- your pick: "decision_tree", "random_forest" or "knn"

champion = searches[CHAMPION].best_estimator_
champion_cv_mae = -searches[CHAMPION].best_score_
champion_test_mae = mean_absolute_error(y_test, champion.predict(X_test))
print("Champion:", CHAMPION, searches[CHAMPION].best_params_)
print("CV MAE $" + format(round(champion_cv_mae), ","), "| test MAE $" + format(round(champion_test_mae), ","),
      "| baseline $" + format(round(baseline_mae), ","))

# Part 5 · Explain it before you ship it (15 min)

Lab 2 came back. SHAP splits each prediction into **one contribution per feature**, measured from the average
prediction (the **base value**): `base value + SHAP(year) + SHAP(mileage) + SHAP(make) + SHAP(model) = prediction`.

The helper below is the **same code `predict-shap` runs in production**. Trees get exact, fast SHAP (`TreeExplainer`);
KNN gets a slower, sampled estimate (`KernelExplainer`). One-hot columns are added back up, so you see 4 features, not 150.

In [ ]:
def original_feature(transformed_name):
    name = transformed_name.split("__", 1)[-1]
    for col in ["make", "model"]:
        if name.startswith(col + "_"):
            return col
    for col in ["year", "mileage"]:
        if name == col or name.startswith(col):
            return col
    return name


def explain_rows(pipe, X, background):
    pre = pipe.named_steps["pre"]
    est = pipe.named_steps["model"]
    Xt = pre.transform(X)
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    names = list(pre.get_feature_names_out())
    if hasattr(est, "tree_") or hasattr(est, "estimators_"):
        explainer = shap.TreeExplainer(est)
        values = explainer.shap_values(Xt, check_additivity=False)
        base = float(np.ravel(explainer.expected_value)[0])
    else:
        Bt = pre.transform(background)
        if hasattr(Bt, "toarray"):
            Bt = Bt.toarray()
        summary = shap.kmeans(Bt, min(10, len(Bt)))
        explainer = shap.KernelExplainer(est.predict, summary)
        values = explainer.shap_values(Xt, nsamples=100, silent=True)
        base = float(np.ravel(explainer.expected_value)[0])
    values = np.asarray(values)
    shap_columns = []
    for feature in FEATURES:
        shap_columns.append("shap_" + feature)
    grouped = pd.DataFrame(0.0, index=X.index, columns=shap_columns)
    for j in range(len(names)):
        grouped["shap_" + original_feature(names[j])] += values[:, j]
    return base, grouped


background = X_train.sample(min(50, len(X_train)), random_state=0)
base_value, test_shap = explain_rows(champion, X_test, background)
check = base_value + test_shap.sum(axis=1) - champion.predict(X_test)
print("Base value (average prediction): $" + format(round(base_value), ","))
print("Largest gap between base + SHAP and the prediction: $" + str(round(float(np.abs(check).max()), 2)))

**Which features drive your model's prices?** Mean absolute SHAP = how many dollars each feature moves a typical prediction.

In [ ]:
importance = test_shap.abs().mean().sort_values()
importance.index = importance.index.str.replace("shap_", "")
importance.plot(kind="barh", color="#6D28A3")
plt.xlabel("Mean |SHAP| (dollars)")
plt.title("What drives " + CHAMPION + "'s prices (test cars)")
plt.show()

**One car, explained.** The waterfall starts at the base value and adds each feature's push until it reaches the
prediction. Change `CAR` to look at another test car.

In [ ]:
def waterfall(base, shap_row, feature_row):
    explanation = shap.Explanation(values=np.array(shap_row, dtype=float), base_values=base,
                                   data=np.array(feature_row, dtype=object), feature_names=FEATURES)
    shap.plots.waterfall(explanation, show=True)


CAR = 0
car_features = X_test.iloc[CAR]
print("Actual price: $" + format(round(y_test.iloc[CAR]), ","), "| predicted: $" + format(round(champion.predict(X_test.iloc[[CAR]])[0]), ","))
waterfall(base_value, test_shap.iloc[CAR].values, car_features.values)

# Part 6 · Ship it (25 min)

## 6a. Save the model and its model card
`model.joblib` is your whole pipeline, frozen. `model_card.json` is its label: what it is, how good it was, which
library versions made it, and which cars it trained on (so production can tell **new** cars from cars it has seen).

In [ ]:
from datetime import datetime, timezone

joblib.dump(champion, "/content/model.joblib")
best_params = {}
for key in searches[CHAMPION].best_params_:
    best_params[key] = str(searches[CHAMPION].best_params_[key])
card = {
    "model_type": type(champion.named_steps["model"]).__name__,
    "champion": CHAMPION,
    "best_params": best_params,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "cv_mae": round(float(champion_cv_mae), 2),
    "test_mae": round(float(champion_test_mae), 2),
    "baseline_mae": round(float(baseline_mae), 2),
    "train_post_ids": list(model_ready.loc[X_train.index, "post_id"].astype(str)),
}
with open("/content/model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print("Saved /content/model.joblib and /content/model_card.json (see the Files panel on the left)")
card_to_show = dict(card)
card_to_show.pop("train_post_ids")
print(json.dumps(card_to_show, indent=2))

## 6b. Get `predict-shap` into your pipeline: Sync fork
Dr. Wanik added a new Cloud Function, **`predict-shap`**, to the course repo. Pull it into yours:
1. Open your fork on GitHub → **Sync fork → Update branch** (the same steps as the Sync-fork video).
2. **Actions** tab: a new workflow, **Deploy predict-shap**, runs by itself. Wait for the green check (3 to 5 minutes:
   it installs SHAP).

Until you upload a model, `predict-shap` politely does nothing (it reports `"status": "no_model"`).

## 6c. Upload your model to your bucket
Type your project ID and bucket (the same ones as Lab 3), then run. Pick your **Google Cloud Gmail** in the pop-up.
No Google Cloud yet? Set `UPLOAD = False`: you can still do Part 7 in demo mode.

In [ ]:
UPLOAD = True
PROJECT_ID = "YOUR-PROJECT-ID"     # e.g. "myscrapers-dww05002-f26-510015"
BUCKET_NAME = "YOUR-PROJECT-ID"    # usually the same as your project ID

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    bucket.blob("models/model.joblib").upload_from_filename("/content/model.joblib")
    bucket.blob("models/model_card.json").upload_from_filename("/content/model_card.json")
    print("Uploaded to gs://" + BUCKET_NAME + "/models/  -> predict-shap uses it at the next :25")

## 6d. Run it now (Cloud Shell)
Instead of waiting for :25, open **Cloud Shell** and run it yourself (paste your own names first if Cloud Shell restarted):
```bash
gcloud config set project YOUR-PROJECT-ID
gcloud scheduler jobs run predict-shap-hourly --location=us-central1
```
Wait about a minute, then look at what it wrote:
```bash
gcloud storage cat "gs://YOUR-BUCKET/structured/preds_lab4/preds_master.csv" | head -5
gcloud storage cat "gs://YOUR-BUCKET/structured/preds_lab4/shap_master.csv" | head -5
```
Nothing there? Read its log and paste it into Claude:
```bash
gcloud functions logs read predict-shap --region=us-central1 --gen2 --limit=20
```

# Part 7 · Read production (10 min)

Your pipeline now saves two files every hour, one row per car:
- `structured/preds_lab4/preds_master.csv`: the car, the **predicted** price, the actual asking price, the error,
  `seen_in_training` (did your model train on this car?), and which model version made the prediction
- `structured/preds_lab4/shap_master.csv`: the **SHAP values** behind each prediction, plus the base value

The next cell reads them from your bucket. No upload in 6c? It switches to **demo mode** and builds the same two
tables from your test cars, so you can still do this part.

In [ ]:
if UPLOAD:
    preds = pd.read_csv(io.BytesIO(bucket.blob("structured/preds_lab4/preds_master.csv").download_as_bytes()), dtype={"post_id": str})
    prod_shap = pd.read_csv(io.BytesIO(bucket.blob("structured/preds_lab4/shap_master.csv").download_as_bytes()), dtype={"post_id": str})
    latest = card["trained_at"]
    preds = preds[preds["model_version"].astype(str) == latest]
    prod_shap = prod_shap[prod_shap["model_version"].astype(str) == latest]
    print("Production predictions from this model:", len(preds))
else:
    print("DEMO MODE: building the production tables from your test cars")
    preds = model_ready.loc[X_test.index, ["post_id", "year", "mileage", "make", "model"]].copy()
    preds["actual_price"] = y_test
    preds["pred_price"] = champion.predict(X_test).round(2)
    preds["abs_error"] = (preds["pred_price"] - preds["actual_price"]).abs()
    preds["seen_in_training"] = False
    prod_shap = test_shap.copy()
    prod_shap["post_id"] = preds["post_id"]
    prod_shap["base_value"] = base_value
    prod_shap["pred_price"] = preds["pred_price"]

preds[["post_id", "year", "make", "model", "mileage", "actual_price", "pred_price", "abs_error", "seen_in_training"]].tail(10)

**Honest scoring.** Production predicts every car, including the ones your model trained on. Cars it has
**seen** look great (it memorized them); cars it has **not** seen are the real test. Compare:

In [ ]:
summary = preds.groupby("seen_in_training")["abs_error"].agg(["count", "mean"]).round(0)
summary.columns = ["cars", "mean abs error ($)"]
summary

**Explain a production prediction.** Pick a car by row number (the newest car is `-1`) and see why your model
priced it that way, straight from the SHAP values production saved.

In [ ]:
ROW = -1
shap_row = prod_shap.iloc[ROW]
car_row = preds[preds["post_id"] == shap_row["post_id"]].iloc[0]
print(car_row["year"], car_row["make"], car_row["model"], "|", car_row["mileage"], "miles")
print("Asking $" + format(round(car_row["actual_price"]), ","), "| your model says $" + format(round(car_row["pred_price"]), ","))
waterfall(float(shap_row["base_value"]),
          [shap_row["shap_year"], shap_row["shap_mileage"], shap_row["shap_make"], shap_row["shap_model"]],
          [car_row["year"], car_row["mileage"], car_row["make"], car_row["model"]])

**Underpriced or overpriced?** A big negative gap (asking far below your model's price) could be a bargain, or a
car with a problem the four features cannot see (a rebuilt title, say). Which is it? Open the ad and check.

In [ ]:
preds["gap"] = preds["actual_price"] - preds["pred_price"]
preds.sort_values("gap")[["post_id", "year", "make", "model", "mileage", "actual_price", "pred_price", "gap"]].head(5)

# Part 8 · Wrap up (5 min)

## Three short answers (1 or 2 sentences each)
1. Which model family won, and by how much did it beat the baseline tree (in dollars of MAE)?
2. Which feature drives your model's prices the most, and does that make sense for used cars?
3. Your model's error on cars it has **not** seen vs cars it **has** seen: what does the gap tell you?

*Your answers:*

---

**Submit:** File → Download → `.ipynb`, then upload it to **Submit Lab 4** on HuskyCT.
**Leave `predict-shap` running.** Every hour it predicts and explains the newest cars, and your midterm can chart how
your model's errors and explanations move over time.